# 04 · E1: the closure ladder on the real worm

**Test of claim C2 (computational closure).** Fit rungs L0–L4 and the black box B to the signal-propagation atlas, then score each on **held-out interventional responses**.

* **Primary protocol:** 5-fold cross-validation over (responder, stimulus) pairs. Each stimulus has its own efficacy parameter, because actuator expression differs across animals.
* **Secondary protocol (stricter):** hold out whole stimulated neurons.

Notebook 02 explains why the primary protocol is pairs: with whole-stimulus holdout, every rung predicted worse than zero on synthetic worms.

| Prediction | Pattern |
|---|---|
| Computationalist | accuracy saturates at some L* ≤ L2; deeper rungs add < 5% of the noise ceiling |
| Organismic (Brette) | gains continue into L3–L4; residual error concentrates in peptide- and state-dominated neurons |
| Simulable-but-not-closed | B beats every mechanistic rung |

**Order of operations:** (1) Notebook 02 has validated the pipeline on synthetic worms with this wiring. (2) Write the pre-registration (§1) **before** running §2. (3) Fit. (4) Apply the registered rule unchanged. Exploratory analyses (§5) are labelled as such.

**Runtime and modes** (see the setup cell's `MODE`):

* `full`: all 300 neurons; this is the registered E1 test.
* `laptop`: the 120 most-sampled neurons, one overnight run on an M1 CPU.
* `smoke`: checks the code only.

A sub-network omits signal routed through the excluded neurons, so a `laptop` result is a **preliminary** E1, not the registered test. Each mode has its own pre-registration and checkpoint files. Checkpoints make every run resumable: if the run is interrupted, just re-run the notebook.

In [1]:
#@title Setup — run first (works locally on a Mac/PC and on Colab)
import os, sys, subprocess
REPO_URL   = "https://github.com/aslansd/Closure-Bench.git"  # Colab only: clone from here
DRIVE_PATH = "/content/drive/MyDrive/Closure-Bench"          # Colab only: ...or use a Drive copy
IN_COLAB = "google.colab" in sys.modules

def _find_package():
    # 1) explicit location, 2) the repo this notebook lives in, 3) Colab clone / Drive
    cands = [os.environ.get("CLOSUREBENCH_ROOT", ""), ".", "..", "/content/Closure-Bench", DRIVE_PATH]
    for root in cands:
        if root and os.path.isdir(os.path.join(root, "closurebench")):
            return os.path.abspath(root)
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wormneuroatlas", "optax"], check=False)
        if subprocess.run(["git", "clone", "-q", REPO_URL, "/content/Closure-Bench"]).returncode == 0:
            return "/content/Closure-Bench"
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.isdir(os.path.join(DRIVE_PATH, "closurebench")):
            return DRIVE_PATH
    raise RuntimeError("closurebench not found. Locally: open this notebook from Closure-Bench/notebooks/, "
                       "or set the environment variable CLOSUREBENCH_ROOT to the Closure-Bench folder.")

ROOT = _find_package()
sys.path.insert(0, ROOT)
try:
    import jax, optax, wormneuroatlas  # noqa: F401
except ImportError as e:
    raise ImportError(f"{e}. Locally, create the environment first: see README.md "
                      "('Run locally'), then select the 'Python (closurebench)' kernel.") from None
RESULTS = os.environ.get("CLOSUREBENCH_RESULTS", os.path.join(ROOT, "results"))
os.makedirs(RESULTS, exist_ok=True)
from closurebench import config as CBC
MODE = CBC.get_mode()        # 'smoke' | 'laptop' | 'full'  (env CLOSUREBENCH_MODE, default: full on GPU, laptop on CPU)
print("closurebench:", ROOT); print("results:", RESULTS); print(CBC.describe_machine()); print("MODE:", MODE)

closurebench: /Users/asataryd/Documents/ENI-Projects/Aslan/NeuroAILab/Githubs/Closure-Bench-Mac
results: /Users/asataryd/Documents/ENI-Projects/Aslan/NeuroAILab/Githubs/Closure-Bench-Mac/results
Darwin arm64 | python 3.11.5 | jax 0.10.2 on cpu (cpu) | CPU cores: 8
MODE: laptop


In [2]:
import json, hashlib, datetime, numpy as np, matplotlib.pyplot as plt, jax
from closurebench import data as D, ladder as lad, metrics as M
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# ---- configuration by MODE (set in the setup cell; override with env CLOSUREBENCH_MODE) ----
#   smoke : 41-neuron sub-network, 30 steps: checks the code path, numbers meaningless
#   laptop: 120 most-sampled neurons, 5 folds; one overnight run on an M1 CPU (see estimate below)
#   full  : all 300 neurons (the registered E1 test); GPU, or a weekend on a laptop CPU
FAST = MODE == "smoke"
CONFIG = dict(levels=["L0", "L1", "L2", "L3", "L4", "B"], k_folds=5, steps=500, lr=2e-2, protocol="pairs",
              equal_budget=True, restarts=3, fold_seed=0, threshold=0.05, n_boot=2000, tie_classes=True,
              n_neurons=300, sim=lad.SimConfig().__dict__)
# steps = per chain stage; with equal_budget every rung gets steps x 4 total optimisation steps
if MODE == "laptop":
    CONFIG.update(steps=250, n_neurons=120)
if MODE == "smoke":
    CONFIG.update(k_folds=2, steps=30, n_boot=200, restarts=2, n_neurons=41)
ds_path = os.path.join(RESULTS, "atlas_dataset.npz")
if not os.path.exists(ds_path):
    D.load_atlas_dataset().save(ds_path)       # same as Notebook 03
ds = D.Dataset.load(ds_path)
if CONFIG["n_neurons"] < ds.N:   # most-sampled sub-network (as responder + as stimulated neuron)
    mk = ds.mask()[0]
    score = mk.sum(1).astype(float); score[ds.stim] += mk.sum(0)
    ds = D.subset(ds, np.sort(np.argsort(-score)[:CONFIG["n_neurons"]]))
print(ds.summary()); print("MODE:", MODE)

Dataset: N=120 neurons, S=120 stimulated neurons, strains=('wt', 'unc31')
  chemical synapses: 1095  gap-junction pairs: 277  peptidergic pairs: 9125
  signs: +163 / -200 / unknown 732 (on existing synapses)
  wt: 13706 observed pairs, median trials 5
  unc31: 6199 observed pairs, median trials 1
MODE: laptop


## 1 · Pre-registration (run once, before fitting)

This cell writes the analysis plan, the decision rule, and a hash of the exact data to `results/E1_preregistration.json`, with a timestamp. It refuses to overwrite an existing registration. Commit the file to GitHub before running §2; the commit time is your public timestamp.

In [3]:
TAG = "" if MODE == "full" else f"_{MODE}"
prereg_path = os.path.join(RESULTS, f"E1_preregistration{TAG}.json")
data_hash = hashlib.sha256(np.ascontiguousarray(ds.mean).tobytes() + np.ascontiguousarray(ds.n).tobytes()).hexdigest()
plan = {
  "experiment": "E1 closure ladder", "mode": MODE, "registered_at": datetime.datetime.utcnow().isoformat() + "Z",
  "data": {"source": ds.meta.get("source"), "sha256": data_hash, "N": int(ds.N), "S": int(ds.S), "strains": list(ds.strains)},
  "config": CONFIG,
  "primary_metric": "pooled FEVE on held-out (responder, stimulus) pairs, k-fold CV over pairs, all observed pairs, both strains",
  "fitting": "nested warm-start chain L1->L4; L1 best of `restarts` initialisations; equal total optimisation budget for all rungs",
  "decision_rule": "L* = lowest rung L such that (i) for every deeper rung L', the 97.5th percentile of the bootstrap "
                   "(resampling stimulated neurons) distribution of FEVE(L') - FEVE(L) is < threshold (0.05), and "
                   "(ii) the 2.5th percentile of FEVE(L) is > 0. If none qualifies: no closure within the ladder.",
  "blackbox_rule": "B beats L* if the 2.5th percentile of FEVE(B) - FEVE(L*) > 0.05 (outcome C signature).",
  "secondary": ["FEVE restricted to significant pairs (atlas q<0.05)", "FEVE per strain", "unc31-specific gain of L3 over L2",
                "whole-stimulus holdout protocol (same rule)"],
  "validation": "synthetic recovery (Notebook 02) must be reported alongside the result",
  "exploratory": ["residual structure vs peptidergic in-degree"],
}
if os.path.exists(prereg_path):
    old = json.load(open(prereg_path))
    assert old["data"]["sha256"] == data_hash, "Data changed since registration: do not proceed without a new, documented registration."
    print("Existing registration found (", old["registered_at"], ") - using it unchanged."); plan = old
else:
    json.dump(plan, open(prereg_path, "w"), indent=2); print("Registered ->", prereg_path)
print(json.dumps(plan["decision_rule"], indent=1))

Registered -> /Users/asataryd/Documents/ENI-Projects/Aslan/NeuroAILab/Githubs/Closure-Bench-Mac/results/E1_preregistration_laptop.json
"L* = lowest rung L such that (i) for every deeper rung L', the 97.5th percentile of the bootstrap (resampling stimulated neurons) distribution of FEVE(L') - FEVE(L) is < threshold (0.05), and (ii) the 2.5th percentile of FEVE(L) is > 0. If none qualifies: no closure within the ladder."


### Runtime estimate for this machine (primary + secondary protocol)

In [4]:
c = CONFIG
_ = CBC.estimate_runtime(ds, levels=c["levels"], k=c["k_folds"], steps=c["steps"], restarts=c["restarts"],
                         n_runs=1 if FAST else 2)

rung    ms/step  compile s  steps (all folds)   minutes
L0         87.0        1.7               5000       7.4
L1        239.9        1.4               7500      30.5
L2        258.3        1.3               3750      16.4
L3        366.9        1.8               2500      15.6
L4        348.4        2.1               1250       7.4
B         140.6        3.3               5000      12.0
estimated total: 3.0 h  (2 runs)  -- rough (+/-50%); checkpoints make it resumable


## 2 · Fit the ladder (cross-validated, held-out interventional responses)

How the rungs are fitted:

* **Nested chain:** L2–L4 are warm-started from the rung below, with the new mechanism switched nearly off.
* **Restarts:** L1 is the best of 3 initializations.
* **Equal budget:** every rung (including L0 and B) gets the same total number of optimization steps.
* **Tying:** bilateral homologs share parameters.

On synthetic worms, dropping equal budgets and restarts made the rule pick rungs one or two levels too deep (Notebook 02).

In [5]:
cfg = lad.SimConfig(**plan["config"]["sim"])
preds, params, dl = M.run_crossvalidated_ladder(
    ds, levels=plan["config"]["levels"], k=plan["config"]["k_folds"], steps=plan["config"]["steps"],
    lr=plan["config"]["lr"], cfg=cfg, seed=plan["config"]["fold_seed"], tie_classes=plan["config"]["tie_classes"],
    protocol=plan["config"]["protocol"], equal_budget=plan["config"]["equal_budget"], restarts=plan["config"]["restarts"],
    checkpoint=os.path.join(RESULTS, f"e1_checkpoints{TAG}"), log=None)
np.savez_compressed(os.path.join(RESULTS, f"E1_heldout_predictions{TAG}.npz"), **preds)

## 3 · Registered decision

In [6]:
levels = plan["config"]["levels"]
point, boots = M.bootstrap_feve(preds, ds, levels, n_boot=plan["config"]["n_boot"])
dec = M.closure_decision(point, boots, threshold=plan["config"]["threshold"])
print(M.format_decision(dec))

rung   FEVE(held-out)   max upper-CI gain of any deeper rung
L0        0.295         +0.088
L1        0.298         +0.028
L2        0.310         +0.012
L3        0.301         +0.006
L4        0.304            —
=> closed level L* = L1
   B - L*: -0.782 [-1.112, -0.545]  -> black box beats L*: False


## 4 · Compression cost of computation

Each rung is plotted at its description length (free parameters, after tying) against its held-out accuracy. Closure shows up as the point where the curve stops rising.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
for L in levels:
    npar, nstate = dl[L]; lo, hi = np.percentile(boots[L], [2.5, 97.5])
    ax.errorbar(npar, point[L], yerr=[[point[L] - lo], [hi - point[L]]], fmt="o" if L != "B" else "s",
                capsize=3, color="k" if L != "B" else "tab:red")
    ax.annotate(f"{L}\n({nstate} states)", (npar, point[L]), textcoords="offset points", xytext=(6, -14), fontsize=8)
wired = [L for L in levels if L != "B"]
ax.plot([dl[L][0] for L in wired], [point[L] for L in wired], "k-", lw=0.8)
ax.axhline(1, ls=":", c="0.5"); ax.text(ax.get_xlim()[0], 1.0, " noise ceiling", va="bottom", fontsize=8, color="0.4")
ax.set_xscale("log"); ax.set_xlabel("description length (free parameters)"); ax.set_ylabel("held-out FEVE")
ax.set_title("Compression cost of computation (E1)"); plt.tight_layout()
plt.savefig(os.path.join(RESULTS, "E1_compression_curve.png"), dpi=200); plt.show()

/var/folders/rk/0gpvj_jn4sz9kf7tyv7brbm00000gn/T/ipykernel_80059/3045392534.py:11: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  ax.set_title("Compression cost of computation (E1)"); plt.tight_layout()


## 5 · Secondary and exploratory analyses

In [8]:
atlas = D.open_atlas()
q = atlas.get_signal_propagation_q("wt")
ids_full = list(atlas.neuron_ids); idx = [ids_full.index(n) for n in ds.ids]
qsub = q[np.ix_(idx, [idx[j] for j in ds.stim])]
sig_mask = np.nan_to_num(qsub < 0.05).astype(bool)
print("Secondary: FEVE on significant pairs only (q<0.05)")
p_s, b_s = M.bootstrap_feve(preds, ds, levels, n_boot=plan["config"]["n_boot"], extra_mask=sig_mask)
for L in levels:
    print(f"  {L}: {p_s[L]:.3f}  [{np.percentile(b_s[L],2.5):.3f}, {np.percentile(b_s[L],97.5):.3f}]")
print("\nSecondary: FEVE per strain")
for s in ds.strains:
    print(f"  {s:6s}", "  ".join(f"{L}={M.feve(preds[L], ds, strain=s):.3f}" for L in levels))
# unc31-specific: only rungs with a peptide channel can predict WT != unc31
print("\nSecondary: does L3 predict the WT-unc31 difference better than L2 (which predicts zero difference)?")
both = (ds.n[0] >= 1) & (ds.n[1] >= 1) & ds.mask()[0]
obs_d = (ds.mean[0] - ds.mean[1])[both]; nv = (ds.var_mean[0] + ds.var_mean[1])[both]
for L in ["L2", "L3", "L4"]:
    pd_ = (preds[L][0] - preds[L][1])[both]
    print(f"  {L}: SSE on difference minus noise = {((pd_ - obs_d) ** 2).sum() - nv.sum():.4f}")

*This version of the NeuroAtlas does not include the CAN neurons. This will be fixed soon.
Secondary: FEVE on significant pairs only (q<0.05)
  L0: 0.173  [0.127, 0.232]
  L1: 0.196  [0.149, 0.244]
  L2: 0.204  [0.157, 0.254]
  L3: 0.201  [0.149, 0.258]
  L4: 0.201  [0.150, 0.258]
  B: 0.084  [0.022, 0.136]

Secondary: FEVE per strain
  wt     L0=0.288  L1=0.268  L2=0.275  L3=0.288  L4=0.292  B=0.327
  unc31  L0=0.310  L1=0.360  L2=0.384  L3=0.328  L4=0.330  B=-2.149

Secondary: does L3 predict the WT-unc31 difference better than L2 (which predicts zero difference)?
  L2: SSE on difference minus noise = 0.4558
  L3: SSE on difference minus noise = 1.1622
  L4: SSE on difference minus noise = 1.1601


In [9]:
# Exploratory (not registered): where does the best wired rung still fail?
from scipy.stats import spearmanr
best = max([L for L in levels if L != "B"], key=lambda L: point[L])
mm = ds.mask()
res = np.where(mm, (preds[best] - ds.mean) ** 2 - ds.var_mean, np.nan)
per_neuron = np.nanmean(np.nanmean(res, axis=0), axis=1)   # (N,)
ok = np.isfinite(per_neuron) & (mm.sum((0, 2)) >= 10)
pep_in = (ds.pep > 0).sum(1); syn_in = (ds.chem > 0).sum(1) + (ds.gap > 0).sum(1)
for name, x in [("peptidergic in-degree", pep_in), ("synaptic in-degree", syn_in)]:
    r, p = spearmanr(x[ok], per_neuron[ok])
    print(f"{best} residual vs {name:22s}: rho={r:+.2f} p={p:.3g} (n={ok.sum()})")
worst = np.argsort(-np.where(ok, per_neuron, -np.inf))[:15]
print("neurons with largest unexplained responses:", ", ".join(ds.ids[worst]))

L2 residual vs peptidergic in-degree : rho=-0.05 p=0.564 (n=120)
L2 residual vs synaptic in-degree    : rho=-0.00 p=0.993 (n=120)
neurons with largest unexplained responses: M2R, M2L, RMDR, M1, AIZR, M5, ADER, URXL, BAGR, BAGL, ASIR, I3, AVKL, I4, RMFR


In [10]:
# Secondary protocol: whole stimulated neurons held out (stricter; expect lower FEVE for every rung)
RUN_STIMULUS_HOLDOUT = not FAST
if RUN_STIMULUS_HOLDOUT:
    c = plan["config"]
    preds_s, _, _ = M.run_crossvalidated_ladder(ds, levels=c["levels"], k=c["k_folds"], steps=c["steps"], lr=c["lr"],
        cfg=cfg, seed=c["fold_seed"], protocol="stimuli", equal_budget=c["equal_budget"], restarts=c["restarts"],
        checkpoint=os.path.join(RESULTS, f"e1_checkpoints_stimuli{TAG}"), log=None)
    p_st, b_st = M.bootstrap_feve(preds_s, ds, levels, n_boot=c["n_boot"])
    print(M.format_decision(M.closure_decision(p_st, b_st, threshold=c["threshold"])))

rung   FEVE(held-out)   max upper-CI gain of any deeper rung
L0        0.261         -0.044
L1        0.108         +0.027
L2        0.116         +0.004
L3        0.091         +0.004
L4        0.091            —
=> closed level L* = L0
   B - L*: -0.965 [-1.530, -0.603]  -> black box beats L*: False


In [11]:
summary = {"preregistration": os.path.basename(prereg_path), "L_star": dec["L_star"],
           "feve": {L: float(point[L]) for L in levels},
           "ci95": {L: [float(np.percentile(boots[L], 2.5)), float(np.percentile(boots[L], 97.5))] for L in levels},
           "description_length": {L: [int(dl[L][0]), int(dl[L][1])] for L in levels},
           "blackbox_beats_Lstar": dec.get("blackbox_beats_Lstar")}
json.dump(summary, open(os.path.join(RESULTS, f"E1_result{TAG}.json"), "w"), indent=2)
print(json.dumps(summary, indent=1))

{
 "preregistration": "E1_preregistration_laptop.json",
 "L_star": "L1",
 "feve": {
  "L0": 0.2950353806808731,
  "L1": 0.29760426880054747,
  "L2": 0.30971020454334264,
  "L3": 0.30055211749785526,
  "L4": 0.3040489002791815,
  "B": -0.46411378374717027
 },
 "ci95": {
  "L0": [
   0.21844911203396206,
   0.41747128706065
  ],
  "L1": [
   0.21474721840114536,
   0.43468228218183974
  ],
  "L2": [
   0.227605540946307,
   0.4467036624422745
  ],
  "L3": [
   0.21537187806157684,
   0.43569522212655015
  ],
  "L4": [
   0.2189824353238117,
   0.4407809735571026
  ],
  "B": [
   -0.70120540318992,
   -0.30790150488455387
  ]
 },
 "description_length": {
  "L0": [
   246,
   120
  ],
  "L1": [
   1244,
   240
  ],
  "L2": [
   1537,
   360
  ],
  "L3": [
   1684,
   480
  ],
  "L4": [
   1831,
   600
  ],
  "B": [
   20441,
   3840
  ]
 },
 "blackbox_beats_Lstar": false
}


## 6 · Interpreting the outcome (proposal §5)

| Result | Verdict |
|---|---|
| L* ∈ {L1, L2}, and B does not beat it | C2 supported here; proceed to E2 (substrate transfer of L*) and E3 (program vs hardware) |
| L* ∈ {L3, L4}, or no closure in the ladder | Brette-style outcome B: software-like, but closure needs extrasynaptic or self-modifying state |
| B beats every rung | outcome C: simulable but not closed at an interpretable level |

**Limits of this first pass:**

* Targets are window-averaged amplitudes, not time courses, so rungs that differ mainly in temporal shape are under-tested. Next step: fit the atlas kernels or the raw traces.
* The surrogate ladder is dimensionless; E2 re-expresses L* in Brian2 and Jaxley.
* Power to separate L2 from L3 is limited by unc-31 coverage (Notebook 03 §4). Notebook 02's power analysis says how much more data would be needed.

---

### Result of the `laptop` run (2026-10-04; 120 most-sampled neurons; preliminary)

| rung | held-out FEVE [95% CI] | free parameters | training loss (5 folds) |
|---|---|---|---|
| L0 | 0.295 [0.22, 0.42] | 246 | 0.92 |
| L1 | 0.298 [0.21, 0.43] | 1,244 | 0.85 |
| L2 | 0.310 [0.23, 0.45] | 1,537 | 0.83 |
| L3 | 0.301 [0.22, 0.44] | 1,684 | 0.81 |
| L4 | 0.304 [0.22, 0.44] | 1,831 | 0.81 |
| B | −0.464 [−0.70, −0.31] | 20,441 | 0.51–0.60 |

The noise floor of the training loss is ≈0.51.

**Registered decision: L* = L1**, and the black box does not beat it. What the numbers show beyond the decision:

1. **Every mechanistic rung ties at ≈0.30, and the linear wiring-only L0 is as good as L1–L4.** Rule (i) picked L1 over L0 only because L0's upper CI for the gain of a deeper rung (+0.088) exceeded 0.05; the point estimates are equal. Under the stricter whole-stimulus protocol, L0 is best (0.26 vs 0.09–0.12) and L* = L0.
2. **The black box is not the answer:** it fits the training data down to the noise floor yet generalizes worst. This is not outcome C.
3. **The mechanistic rungs underfit.** Their training loss (0.81–0.92) stays far above the noise floor (0.51). Continuing the fold-1 L1 fit for 1,000 more steps lowered its training loss (0.857 → 0.815) but *reduced* held-out FEVE (0.244 → 0.149). The ceiling is set by the model class and the amplitude targets, not by the training budget.
4. **No peptidergic signal.** L3's peptide channel predicts the WT − unc-31 difference *worse* than assuming no difference (SSE 1.16 vs 0.46), consistent with Notebook 03 §4 (CI including zero).
5. **Residuals do not follow peptidergic in-degree** (ρ = −0.05), so there is no support for the organismic residual pattern. The least-explained responders are mostly **pharyngeal** (M1, M2L/R, M5, I3, I4) or O₂/CO₂-sensing and head-motor neurons (BAGL/R, URXL, RMDR, AIZR). The pharyngeal system connects to the somatic one through a single neuron pair (RIPL/R), so wiring predicts little there. These neurons are candidates for extrasynaptic or humoral coupling.

**Verdict at this resolution:** with window-averaged amplitudes on 120 neurons, nothing beyond wiring and signs measurably improves prediction of held-out interventions. Combined with Notebook 02 (the test misses L2 and weak L3), this is **"no detectable need for mechanism beyond the wiring", not "closure at L1"**. The 70% of explainable variance that no rung captures is the main open problem. It motivates Notebook 06: time-course targets and the full 300-neuron network.

> **Follow-up (added 2026-10-08, after Notebooks 06–19).**
> * Time-course targets did **not** rescue L2/L3 (06).
> * The deeper rungs are unidentifiable from amplitude data. The real-data statement is the bracket **L0 ≤ closed ≤ L1** (07); under the null-calibrated rule the closed level is L0 (09).
> * The 300-neuron run has not been done (it needs `full` mode).
>
> Point 3 above matters more than it seemed. Notebook 19 found that a slow current added to the fitted L1 improves held-out FEVE by +0.025 [+0.013, +0.039]. Its preferred form is equivalent to a *rescaled L1*, a point inside the L1 family. Further gradient steps lowered held-out FEVE (point 3), so if L1 is under-fitted, the fit is not at the best point of its family, and gradient descent on the training loss does not lead there. Notebook 20 found that E1's 30-s burn-in does not quite reach rest (the fits relax over 10–13 s). Notebook 21 showed that this barely changes E1's predictions (≤ 0.006 FEVE), so the results above stand. Notebook 22 refitted L1 with and without a global slow current: the slow current keeps +0.024 [+0.014, +0.039], whereas 300 further steps on L1 alone neither helped nor hurt (+0.002). E1's nested warm start cannot grow it from "nearly off". Re-applying this notebook's rules with that rung on the ladder (Notebook 23) leaves the verdict unchanged: it beats L1 but not reliably L0.
>
> **The black box result above is strain-specific.** On WT pairs alone, B scores 0.327, above every rung; its −0.46 overall comes from unc-31. Notebook 24 tests whether this is outcome C on wild type.
>
> **L0 above was under-fitted (added 2026-10-09, Notebook 24).** L0 had the same total budget as the other rungs but a single start, with no restarts. Three hundred more steps from its fit here raise its held-out FEVE from 0.295 to **0.375**, above every rung and above the black box. The same treatment moved L1 by only +0.002–0.004. Consequences:
> * The verdict is reinforced: with the refined L0, all three rules, sufficiency included, give **L0**.
> * B's WT lead disappears: it falls behind L0 plus a slow current, −0.016 [−0.059, +0.025], so there is no outcome C.
> * The rung-to-rung gaps in the table above partly reflect unequal fit quality, not mechanism. Notebook 25 refines every rung with an equal budget, checks convergence and re-decides.
>
> **Fair ladder (added 2026-10-10, Notebook 25).** Every rung and the black box were refined with the same extra budget, in two rounds of 300 steps.
> * **All three rules give L0, on both strains and on WT.** Every deeper rung is significantly *worse* than L0: L1 −0.12 [−0.22, −0.03], and L2–L4 similar. L0 reaches **0.421** and is still improving.
> * **The deeper rungs and B over-fit as they train.** Their training FEVE rises while their held-out FEVE falls, and B's training FEVE exceeds 1 (it fits trial noise).
> * B − L0 on WT = −0.22 [−0.31, −0.15]: no outcome C.
> * The table above therefore understates L0 and overstates nothing that changes the verdict.
>
> Notebook 26 asks why richer rungs lose. L1 adds both a nonlinearity and synapse-resolved weights, and Notebook 26 tests each separately.
>
> **Nonlinearity or parameter count? (Notebook 26).** A compact nonlinear model, with L1's sigmoid neurons but L0's three class weights (345 parameters instead of 1124), is still **0.096 worse than L0** [−0.153, −0.049]. So L1 does not lose only by over-fitting its weights. Whether a *linear* model needs synapse-specific weights could not be tested there and is tested cleanly in Notebook 27.
>
> **Synapse-specific weights? (Notebook 27, open).** Fitted cleanly, a linear model with a weight per synapse (`L0w`) beat a clean L0 by +0.076. But that L0 was again under-fitted (0.300), and `L0w` got extra optimization on top of it. Its weights moved only 12–16 % from contact counts, with no sign flips. Notebook 28 converges every model first.